# 01 — Graph Structure

Visualize the vault DAG: notes, dependencies, levels, and execution order.

**Requires:** A vault with `.vault-exec/vault.kv` (run `vault-exec init` first)

In [1]:
// Setup: load vault notes and build graph
import { DenoVaultReader } from "../src/io.ts";
import { parseVault } from "../src/parser.ts";
import { buildGraph, topologicalSort } from "../src/graph.ts";

const VAULT_PATH = "../demo-vault";

const reader = new DenoVaultReader();
const notes = await parseVault(reader, VAULT_PATH);
const graph = buildGraph(notes);
const order = topologicalSort(graph);

console.log(`Vault: ${notes.length} notes, ${graph.edges.size} dependency sets`);
console.log(`Execution order: ${order.join(" → ")}`);

Vault: 22 notes, 22 dependency sets


Execution order: Employees → Department Budget → Engineering Team → Expansion Signals → Intent Sandbox Node → Market Signals → NPS Snapshot → Projects → Active Projects → Biggest Team → Hiring Need → Project Costs → Over Budget → Renewal Watchlist → Revenue Risk Score → Support Tickets → Action Queue → Intent Routing Check → Total Payroll → Unlinked Ideas Backlog → Risk Ops Hub → Upsell Candidates


In [2]:
// Note details: type, level, dependencies
const levels = new Map<string, number>();
const noteSet = new Set(notes.map(n => n.name));

function getLevel(name: string): number {
  if (levels.has(name)) return levels.get(name)!;
  const note = notes.find(n => n.name === name);
  if (!note || note.wikilinks.length === 0) { levels.set(name, 0); return 0; }
  const depLevels = note.wikilinks.filter(w => noteSet.has(w)).map(w => getLevel(w));
  const level = depLevels.length > 0 ? Math.max(...depLevels) + 1 : 0;
  levels.set(name, level);
  return level;
}
notes.forEach(n => getLevel(n.name));

const table = notes.map(n => ({
  name: n.name,
  type: n.frontmatter.code ? 'code' : (n.frontmatter.value ? 'value' : 'unknown'),
  level: levels.get(n.name) ?? 0,
  deps: n.wikilinks.join(', ') || '(none)',
  outputs: ((n.frontmatter.outputs as string[]) ?? []).join(', '),
  compiled: !!n.frontmatter.compiled_at,
})).sort((a, b) => a.level - b.level);

console.table(table);

┌───────┬──────────────────────────┬─────────┬───────┬────────────────────────────────────────────────────────────┬──────────────────────┬──────────┐
│ (idx) │ name                     │ type    │ level │ deps                                                       │ outputs              │ compiled │
├───────┼──────────────────────────┼─────────┼───────┼────────────────────────────────────────────────────────────┼──────────────────────┼──────────┤
│     0 │ "Employees"              │ "value" │     0 │ "(none)"                                                   │ "employees"          │ true     │
│     1 │ "Expansion Signals"      │ "value" │     0 │ "(none)"                                                   │ "expansionSignals"   │ true     │
│     2 │ "Intent Sandbox Node"    │ "value" │     0 │ "(none)"                                                   │ "intentSandbox"      │ true     │
│     3 │ "Market Signals"         │ "value" │     0 │ "(none)"                                     

In [3]:
// Vega-Lite: DAG as a node-link diagram
// Build nodes and edges for visualization
const vegaNodes = notes.map(n => ({
  name: n.name,
  type: n.frontmatter.code ? 'code' : 'value',
  level: levels.get(n.name) ?? 0,
}));

const vegaEdges: Array<{source: string, target: string}> = [];
for (const note of notes) {
  for (const dep of note.wikilinks) {
    if (noteSet.has(dep)) {
      vegaEdges.push({ source: dep, target: note.name });
    }
  }
}

// Display as Vega-Lite layered chart
const spec = {
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  "title": "Vault DAG — Notes by Level",
  "width": 600,
  "height": 300,
  "data": { "values": vegaNodes },
  "mark": { "type": "point", "filled": true, "size": 200 },
  "encoding": {
    "x": { "field": "level", "type": "ordinal", "title": "DAG Level" },
    "y": { "field": "name", "type": "nominal", "title": "Note", "sort": null },
    "color": { "field": "type", "type": "nominal", "scale": { "domain": ["value", "code"], "range": ["#4CAF50", "#2196F3"] } },
    "shape": { "field": "type", "type": "nominal" },
    "tooltip": [{ "field": "name" }, { "field": "type" }, { "field": "level" }]
  }
};

await Deno.jupyter.display({ "application/vnd.vegalite.v5+json": spec }, { raw: true });

In [4]:
// Dependency matrix: which note depends on which
const names = notes.map(n => n.name).sort();
const matrix: Array<{source: string, target: string, depends: number}> = [];
for (const src of names) {
  for (const tgt of names) {
    const note = notes.find(n => n.name === tgt);
    const depends = note?.wikilinks.includes(src) ? 1 : 0;
    matrix.push({ source: src, target: tgt, depends });
  }
}

const matrixSpec = {
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  "title": "Dependency Matrix",
  "width": 400,
  "height": 400,
  "data": { "values": matrix },
  "mark": "rect",
  "encoding": {
    "x": { "field": "source", "type": "nominal", "title": "Depends On" },
    "y": { "field": "target", "type": "nominal", "title": "Note" },
    "color": { "field": "depends", "type": "quantitative", "scale": { "domain": [0, 1], "range": ["#f5f5f5", "#1976D2"] }, "legend": null },
    "tooltip": [{ "field": "target" }, { "field": "source" }, { "field": "depends" }]
  }
};

await Deno.jupyter.display({ "application/vnd.vegalite.v5+json": matrixSpec }, { raw: true });

In [5]:
// Summary stats
const valueNodes = notes.filter(n => n.frontmatter.value).length;
const codeNodes = notes.filter(n => n.frontmatter.code).length;
const maxLevel = Math.max(...[...levels.values()]);
const leafNodes = notes.filter(n => n.wikilinks.length === 0).length;
const avgDeps = notes.reduce((s, n) => s + n.wikilinks.length, 0) / notes.length;

console.log(`\n=== Vault Summary ===`);
console.log(`Total notes:  ${notes.length}`);
console.log(`Value nodes:  ${valueNodes}`);
console.log(`Code nodes:   ${codeNodes}`);
console.log(`Leaf nodes:   ${leafNodes}`);
console.log(`Max depth:    ${maxLevel}`);
console.log(`Avg deps:     ${avgDeps.toFixed(1)}`);
console.log(`Edges:        ${vegaEdges.length}`);


=== Vault Summary ===


Total notes:  22


Value nodes:  11


Code nodes:   11


Leaf nodes:   10


Max depth:    3


Avg deps:     0.9


Edges:        19
